# Pipeline 02: Agente RAG y LLM

Este notebook toma las predicciones generadas (predicciones_riesgo) y los documentos estructurados para indexarlos en PGVector y permitir consultas en lenguaje natural.

In [ ]:
import sys
import os
import importlib
from pathlib import Path
from dotenv import load_dotenv

current_dir = Path.cwd()
ROOT_DIR = current_dir if (current_dir / "src").exists() else current_dir.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

load_dotenv(ROOT_DIR / ".env", override=True)

from src.config import settings
from src.db_utils import create_supabase_engine
import pandas as pd

engine = create_supabase_engine(settings.DATABASE_URL)

## 1. Poblar base vectorial RAG

Indexa en PGVector el resumen del portafolio, políticas de negocio y fichas de créditos.

In [ ]:
import scripts.populate_rag as _populate_rag
importlib.reload(_populate_rag)
run_populate_rag = _populate_rag.main

chunks_indexados = run_populate_rag(reset=True, include_fichas=True)
print(f"Chunks indexados en PGVector: {chunks_indexados}")

rag_counts = pd.read_sql("""
SELECT
    (SELECT COUNT(*) FROM langchain_pg_collection) AS colecciones,
    (SELECT COUNT(*) FROM langchain_pg_embedding) AS embeddings;
""", engine)
display(rag_counts)

## 2. Preguntar al agente RAG + LLM

El agente recupera contexto desde PGVector y llama al LLM para responder con base en los datos indexados.

In [ ]:
from src.rag_agent import RAGAgentPipeline

agent = RAGAgentPipeline(
    db_connection=settings.DATABASE_URL,
    nvidia_api_key=settings.NVIDIA_API_KEY,
)

def preguntar_tumipay(pregunta: str) -> str:
    respuesta = "".join(agent.stream_query(pregunta)).strip()
    print(f"Pregunta:\n{pregunta}\n\nRespuesta del agente:\n{respuesta}")
    return respuesta

pregunta = "¿Cuál es la tasa de mora del portafolio y qué acciones recomiendas para los segmentos de mayor riesgo?"
respuesta = preguntar_tumipay(pregunta)